In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [4]:
df = pd.read_csv("../UCI-dataset/diabetic_data.csv")

In [5]:
df = df.replace("?", np.nan)

In [6]:
df["readmitted_30"] = (
    df["readmitted"] == "<30"
).astype(int)

In [7]:
selected_features = [
    "age",
    "gender",
    "admission_type_id",
    "admission_source_id",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "insulin",
    "change",
    "diabetesMed"
]

In [8]:
X = df[selected_features]
y = df["readmitted_30"]

In [9]:
numerical_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

In [10]:
categorical_features = [
    "age",
    "gender",
    "admission_type_id",
    "admission_source_id",
    "insulin",
    "change",
    "diabetesMed"
]

In [11]:
groups = df["patient_nbr"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (81613, 15)
Testing shape: (20153, 15)


In [12]:
numerical_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [14]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ]
)

In [15]:
logistic_model.fit(X_train, y_train)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


In [16]:
y_pred_lr = logistic_model.predict(X_test)

y_prob_lr = logistic_model.predict_proba(X_test)[:, 1]

In [17]:
lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr)
lr_recall = recall_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)
lr_roc_auc = roc_auc_score(y_test, y_prob_lr)

print("Accuracy:", lr_accuracy)
print("Precision:", lr_precision)
print("Recall:", lr_recall)
print("F1 Score:", lr_f1)
print("ROC-AUC:", lr_roc_auc)

Accuracy: 0.6721579913660497
Precision: 0.15744157441574416
Recall: 0.47605764760576474
F1 Score: 0.2366262276140959
ROC-AUC: 0.6273361020736836


In [18]:
cm_lr = confusion_matrix(y_test, y_pred_lr)

print(cm_lr)

[[12522  5480]
 [ 1127  1024]]


In [19]:
print(classification_report(y_test, y_pred_lr))

              precision    recall  f1-score   support

           0       0.92      0.70      0.79     18002
           1       0.16      0.48      0.24      2151

    accuracy                           0.67     20153
   macro avg       0.54      0.59      0.51     20153
weighted avg       0.84      0.67      0.73     20153



In [20]:
random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=30,
            max_depth=10,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

In [21]:
random_forest_model.fit(X_train, y_train)

print("Random Forest model trained successfully!")

Random Forest model trained successfully!


In [22]:
y_pred_rf = random_forest_model.predict(X_test)

y_prob_rf = random_forest_model.predict_proba(X_test)[:, 1]

In [23]:
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)
rf_roc_auc = roc_auc_score(y_test, y_prob_rf)

print("Accuracy:", rf_accuracy)
print("Precision:", rf_precision)
print("Recall:", rf_recall)
print("F1 Score:", rf_f1)
print("ROC-AUC:", rf_roc_auc)

Accuracy: 0.6601498536198085
Precision: 0.1561768149882904
Recall: 0.49604834960483496
F1 Score: 0.23755983524435045
ROC-AUC: 0.626634361769091


In [24]:
cm_rf = confusion_matrix(y_test, y_pred_rf)

print(cm_rf)

[[12237  5765]
 [ 1084  1067]]


In [25]:
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.92      0.68      0.78     18002
           1       0.16      0.50      0.24      2151

    accuracy                           0.66     20153
   macro avg       0.54      0.59      0.51     20153
weighted avg       0.84      0.66      0.72     20153



In [27]:
%pip install xgboost

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
    --------------------------------------- 0.8/48.9 MB 870.5 kB/s eta 0:00:56
   - -------------------------------------- 1.6/48.9 MB 1.0 MB/s eta 0:00:48
   -- ------------------------------------- 2.6/48.9 MB 1.1 MB/s eta 0:00:41
   -- ------------------------------------- 3.1/48.9 MB 1.2 MB/s eta 0:00:38
   -- ------------------------------------- 3.4/48.9 MB 1.2 MB/s eta 0:00:38
   -- ------------------------------------- 3.7/48.9 MB 1.3 MB/s eta 0:00:37
   --- ------------------------------------ 3.9/48.9 MB 1.3 MB/s eta 0:00:36
   --- ------------------------------------ 4.5/48.9 MB 1.3 MB/s eta 0:00:34
   --- ------------------------------------ 4.5/48.9 MB 1.3 MB/s eta 0:00:34
   --- ------------------------------------ 4.7/48.9 MB 1.1 MB/s eta 0:00:39
   ---- ----------------------------------- 5.0/48.9 MB 1.2 MB/s eta 0:00:39
   ---- ---


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
from xgboost import XGBClassifier

print("XGBoost imported successfully!")

XGBoost imported successfully!


In [29]:
xgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", XGBClassifier(
            n_estimators=50,
            max_depth=4,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            eval_metric="logloss",
            n_jobs=-1
        ))
    ]
)

In [30]:
xgb_model.fit(X_train, y_train)

print("XGBoost model trained successfully!")

XGBoost model trained successfully!


In [31]:
y_pred_xgb = xgb_model.predict(X_test)

y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

In [32]:
xgb_accuracy = accuracy_score(y_test, y_pred_xgb)
xgb_precision = precision_score(y_test, y_pred_xgb)
xgb_recall = recall_score(y_test, y_pred_xgb)
xgb_f1 = f1_score(y_test, y_pred_xgb)
xgb_roc_auc = roc_auc_score(y_test, y_prob_xgb)

print("Accuracy:", xgb_accuracy)
print("Precision:", xgb_precision)
print("Recall:", xgb_recall)
print("F1 Score:", xgb_f1)
print("ROC-AUC:", xgb_roc_auc)

Accuracy: 0.893266511189401
Precision: 0.5
Recall: 0.003719200371920037
F1 Score: 0.007383479464697739
ROC-AUC: 0.6338410614121031


In [33]:
cm_xgb = confusion_matrix(y_test, y_pred_xgb)

print(cm_xgb)

[[17994     8]
 [ 2143     8]]


In [34]:
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.89      1.00      0.94     18002
           1       0.50      0.00      0.01      2151

    accuracy                           0.89     20153
   macro avg       0.70      0.50      0.48     20153
weighted avg       0.85      0.89      0.84     20153



In [36]:
model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy": [
        lr_accuracy,
        rf_accuracy,
        xgb_accuracy
    ],
    "Precision": [
        lr_precision,
        rf_precision,
        xgb_precision
    ],
    "Recall": [
        lr_recall,
        rf_recall,
        xgb_recall
    ],
    "F1 Score": [
        lr_f1,
        rf_f1,
        xgb_f1
    ],
    "ROC-AUC": [
        lr_roc_auc,
        rf_roc_auc,
        xgb_roc_auc
    ]
})

model_comparison

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.672158,0.157442,0.476058,0.236626,0.627336
1,Random Forest,0.660150,0.156177,0.496048,0.237560,0.626634
2,XGBoost,0.893267,0.500000,0.003719,0.007383,0.633841


In [37]:
import joblib

In [38]:
joblib.dump(
    random_forest_model,
    "../models/best_model.pkl"
)

print("Best model saved successfully!")

Best model saved successfully!


In [39]:
priority_data = X_test.copy()

priority_data["Actual_Readmission"] = y_test.values
priority_data["Readmission_Probability"] = y_prob_rf

priority_data.to_csv(
    "../results/patient_predictions.csv",
    index=False
)

print("Patient predictions saved successfully!")

Patient predictions saved successfully!
